In [ ]:
"""
Create NextGen v2.2 hydrofabric geopackage given the IDs used in the Texas case study.
Code and workflow from NGIAB_data_preprocess
https://github.com/CIROH-UA/NGIAB_data_preprocess.git

Written by Quinn Russell
Source code by Josh Cunningham
"""
import json
from typing import List, Union, Tuple
from pathlib import Path
import sqlite3

import geopandas as gpd

In [9]:
# Get reach IDs from TX case study json
with open("travis-county-lo-res-model-partitioned.json", "r", encoding="utf-8") as f:
    models = json.load(f)

reaches = []
for model_id in models["models"]:
    reaches += models["models"][model_id]["model"]["reach_ids"]

In [10]:
intreaches = [int(reach) for reach in reaches]

In [ ]:
# Map NWM IDs to NextGen IDs
with open("hf2.2_ref_hf_map.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)

reach_floats = [float(reach) for reach in reaches]
reverse_map = set()
for ngen_id, nwm_ids in mapping.items():
    for nwm_id in nwm_ids:
        if nwm_id in reach_floats:
            reverse_map.add(ngen_id)

ngen_reaches = list(reverse_map)

wb_ids = [i.replace("cat-", "wb-", 1) if i.startswith("cat-") else i for i in ngen_reaches]

In [ ]:
len(wb_ids)

557

In [15]:
# Useful functions from NGIAB data preprocess's gpkg_utils.py

def create_empty_gpkg(gpkg_arg: Path) -> None:
    """
    Create an empty geopackage with the necessary tables and indices.
    """
    with open("template.sql", encoding="utf-8") as template_file:
        sql_script = template_file.read()

    with sqlite3.connect(gpkg_arg) as conn:
        conn.executescript(sql_script)

def insert_data(con: sqlite3.Connection, table: str, contents: List[Tuple]) -> None:
    """
    Insert data into the specified table.

    Args:
        con (sqlite3.Connection): The database connection.
        table (str): The table name.
        contents (List[Tuple]): The data to be inserted.
    """
    if len(contents) == 0:
        return

    placeholders = ",".join("?" * len(contents[0]))
    con.executemany(f"INSERT INTO '{table}' VALUES ({placeholders})", contents)
    con.commit()

def get_feature_tables(gpkg_arg: Path) -> List[str]:
    """Takes a Path to a geopackage and returns a list of tables containing geometries."""
    sql_query = "SELECT table_name FROM gpkg_contents WHERE data_type='features'"
    with sqlite3.connect(gpkg_arg) as conn:
        tables = conn.execute(sql_query).fetchall()
    tables = [i[0] for i in tables]
    return tables

def create_rtree_table(table: str, con: sqlite3.Connection) -> None:
    """
    Create an rTree table for the specified table.

    Args:
        table (str): The table name.
        con (sqlite3.Connection): The database connection.
    """
    con.execute(
        f'CREATE VIRTUAL TABLE "rtree_{table}_geom" USING rtree("id", "minx", "maxx", "miny", ' +
        '"maxy")'
    )
    con.commit()


def copy_rtree_tables(
    table: str, ids: List[str], source_db: sqlite3.Connection, dest_db: sqlite3.Connection
) -> None:
    """
    Copy rTree tables from source database to destination database.
    This contains the spatial index for the specified table.
    Copying it saves us from having to rebuild the index.
    Args:
        table (str): The table name.
        ids (List[str]): The list of IDs.
        source_db (sqlite3.Connection): The source database connection.
        dest_db (sqlite3.Connection): The destination database connection.
    """
    rtree_table = f"rtree_{table}_geom"

    create_rtree_table(table, dest_db)

    geom_data = source_db.execute(
        f"SELECT * FROM {rtree_table} WHERE id in ({','.join(ids)})"
    ).fetchall()
    insert_data(dest_db, rtree_table, geom_data)

def subset_table(table: str, ids: List[str], hydrofabric: Path, subset_gpkg_name: Path) -> None:
    """
    Subset the specified table from the hydrofabric database and save it to the subset geopackage.

    Args:
        table (str): The table name.
        ids (List[str]): The list of IDs.
        hydrofabric (str): The path to the hydrofabric database.
        subset_gpkg_name (str): The name of the subset geopackage.
    """
    source_db = sqlite3.connect(f"file:{hydrofabric}?mode=ro", uri=True)
    dest_db = sqlite3.connect(subset_gpkg_name)

    table_keys = {"divide-attributes": "divide_id", "lakes": "poi_id"}

    if table == "lakes":
        # lakes subset we get from the pois table which was already subset by water body id
        sql_query = "SELECT poi_id FROM 'pois'"
        contents = dest_db.execute(sql_query).fetchall()
        ids = [str(x[0]) for x in contents]

    if table == "divide-attributes":
        # get the divide ids from the divides that have been subset already
        sql_query = "SELECT divide_id FROM 'divides'"
        contents = dest_db.execute(sql_query).fetchall()
        ids = [str(x[0]) for x in contents]

    if table == "nexus":
        # add the nexuses in the toid column from the flowpaths table
        sql_query = "SELECT toid FROM 'flowpaths'"
        contents = dest_db.execute(sql_query).fetchall()
        new_ids = [str(x[0]) for x in contents]
        ids.extend(new_ids)

    ids = [f"'{x}'" for x in ids]
    key_name = "id"
    if table in table_keys:
        key_name = table_keys[table]
    sql_query = f"SELECT * FROM '{table}' WHERE {key_name} IN ({','.join(ids)})"
    contents = source_db.execute(sql_query).fetchall()

    insert_data(dest_db, table, contents)

    if table in get_feature_tables(hydrofabric):
        fids = [str(x[0]) for x in contents]
        copy_rtree_tables(table, fids, source_db, dest_db)

    dest_db.commit()
    source_db.close()
    dest_db.close()

def add_triggers_to_gpkg(gpkg_arg: Path) -> None:
    """
    Adds geopackage triggers required to maintain spatial index integrity
    """
    with open("triggers.sql", encoding="utf-8") as triggers_file:
        triggers = triggers_file.read()
    with sqlite3.connect(gpkg_arg) as conn:
        conn.executescript(triggers)

def update_geopackage_metadata(gpkg_arg: Path, hydrofabric: Path ) -> None:
    """
    Update the contents of the gpkg_contents table in the specified geopackage.
    """
    # table_name, data_type, identifier, description, last_change, min_x, min_y, max_x, max_y,
    # srs_id
    tables = get_feature_tables(hydrofabric)
    con = sqlite3.connect(gpkg_arg)
    for table in tables:
        min_x = con.execute(f"SELECT MIN(minx) FROM rtree_{table}_geom").fetchone()[0]
        min_y = con.execute(f"SELECT MIN(miny) FROM rtree_{table}_geom").fetchone()[0]
        max_x = con.execute(f"SELECT MAX(maxx) FROM rtree_{table}_geom").fetchone()[0]
        max_y = con.execute(f"SELECT MAX(maxy) FROM rtree_{table}_geom").fetchone()[0]
        srs_id = con.execute(
            f"SELECT srs_id FROM gpkg_geometry_columns WHERE table_name = '{table}'"
        ).fetchone()[0]
        sql_command = (
            "INSERT INTO gpkg_contents (table_name, data_type, identifier, description, " +
            f"last_change, min_x, min_y, max_x, max_y, srs_id) VALUES ('{table}', 'features', " +
            f"'{table}', '', datetime('now'), {min_x}, {min_y}, {max_x}, {max_y}, {srs_id})"
        )
        sql_command = sql_command.replace("None", "NULL")
        con.execute(sql_command)
    con.commit()

    # do some gpkg spec updating
    con.execute("PRAGMA application_id = '0x47504B47'")
    con.execute("PRAGMA user_version = 10200")
    con.commit()

    # update the gpkg_ogr_contents table with table_name and number of features
    for table in tables:
        num_features = con.execute(f"SELECT COUNT(*) FROM '{table}'").fetchone()[0]
        con.execute(
            f"INSERT INTO gpkg_ogr_contents (table_name, feature_count) VALUES ('{table}', " +
            f"{num_features})"
        )

    con.close()


In [16]:
# Useful functions and variables from NGIAB data preprocess's subset.py

subset_tables = [
    "divides",
    "divide-attributes",  # requires divides
    "flowpath-attributes",
    "flowpath-attributes-ml",
    "flowpaths",
    "hydrolocations",
    "nexus",  # depends on flowpaths in some cases e.g. gage delineation
    "pois",  # requires flowpaths
    "lakes",  # requires pois
    "network",
]

def create_subset_gpkg(
    ids: Union[List[str], str],
    hydrofabric: Path,
    output_gpkg_path: Path,
):
    """Subset geopackage given a list of IDs.
    """
    # ids is a list of nexus and wb ids, or a single vpu id
    if not isinstance(ids, list):
        ids = [ids]
    output_gpkg_path.parent.mkdir(parents=True, exist_ok=True)

    create_empty_gpkg(output_gpkg_path)
    for table in subset_tables:
        subset_table(table, ids, hydrofabric, output_gpkg_path)

    add_triggers_to_gpkg(output_gpkg_path)
    update_geopackage_metadata(output_gpkg_path, hydrofabric=hydrofabric)

In [ ]:
create_subset_gpkg(
    wb_ids,
    Path("~/.ngiab/hydrofabric/v2.2/conus_nextgen.gpkg").expanduser(),
    Path("./tx_subset.gpkg").expanduser()
)

In [20]:
gpkg = gpd.read_file("tx_subset.gpkg", layer="divides")
gpkg.head()

,divide_id,toid,type,ds_id,areasqkm,vpuid,id,lengthkm,tot_drainage_areasqkm,has_flowline,geometry
0,cat-2416827,nex-2416828,network,NaN,24.37515,12,wb-2416827,8.526311,39.16305,True,"POLYGON ((-178335 769665, -178275 769725, -178..."
1,cat-2416921,nex-2416829,network,NaN,10.41750,12,wb-2416921,8.068753,65.81565,True,"POLYGON ((-167415 765735, -167355 765525, -167..."
2,cat-2416924,nex-2416921,network,NaN,6.04305,12,wb-2416924,3.848510,14.67990,True,"POLYGON ((-173325 771225, -173535 771075, -173..."
3,cat-2416931,nex-2416932,network,NaN,6.94800,12,wb-2416931,2.771360,22.05360,True,"POLYGON ((-175125 773445, -174975 773295, -174..."
4,cat-2419956,nex-2419864,network,NaN,14.51250,12,wb-2419956,5.907158,62.98830,True,"POLYGON ((-157215 832755, -157065 832845, -156..."
